In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.io as pio

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from utils.equivariance_validation import trial_caches_all
import utils.simulation as sim
import utils.plot_simulation as plot_sim

pio.kaleido.scope.mathjax = None

In [ ]:
#the recorded populations (pixels, AlexNet conv1-5, the visual areas of the three mice) and the
#standard analysis parameters (N = 250 neurons per subsample, d = 20, the data's noise model)
model_df = pd.read_csv( project_root / 'params' / 'model_params.csv', header=0)
params_df = pd.read_csv( project_root / 'params' / 'fixed_params.csv', header=0, index_col=0)

results_simulation = project_root / 'results_extended' / 'simulation'
results_simulation.mkdir(parents=True, exist_ok=True)

### low-dimensional simulations (ED Fig 8, SI Fig 7)
synthetic populations with a known low-dimensional equivariant subspace, one parameter varied per axis
around a model-abiding reference (`sim.grid`), passed through the identical analysis pipeline; 10 seeds
per cell, all 46 leave-two-out pairs, 4 neuron subsamples
per fit. Every (cell, seed) run is cached in `cache/simulation_v3` and skipped afterwards; the first run
takes long (a run of the reference cell is ~1 h on a workstation; on a cluster, one call per array task)

In [ ]:
#the (cell, seed) runs of the benchmark
cells_seeds = [(cell, seed) for cell in sim.grid() for seed in range(10)]
for cell, seed in cells_seeds:
    sim.run_cell_cached(project_root, cell, seed, n_max=4, n_pairs=None)

In [ ]:
#ED Fig 8b-c: per axis the estimated invariant / tuned dimensions, the two decoders in the full space
#and in the estimated subspaces, and the pooled centered CC_norm of the equivariance model (mean +- sem
#over seeds) -> results_extended/simulation/simulation_{axis}_{dims,svm_err,ori_err,cc_norm_equivariance_c}.pdf
runs = plot_sim.load_runs(project_root)
summary = plot_sim.summarize(runs)
summary.to_csv(results_simulation / 'simulation_summary.csv', index=False)
for axis in plot_sim.AXES:
    plot_sim.plot_axis(summary, axis, results_simulation)
summary[['cell', 'n_seeds', 'true_d_inv mean', 'd_inv mean', 'true_d_tun mean', 'd_tun mean', 'cc_norm_equivariance_c mean',
         'svm_err_best mean', 'svm_err_eqv_best mean', 'ori_err mean', 'ori_err_eqv mean']].round(3)

In [ ]:
#SI Fig 7: calibration of the held-out scores against the noiseless ground truth of the generator (the
#data-matched populations of ED Fig 10b are overlaid once results_extended/data_fit/data_fit_sim.csv exists)
#-> simulation_calibration_{ccnorm_vs_corr,ccnorm2_vs_r2,spe_vs_r2}.pdf
plot_sim.plot_calibration(summary, results_simulation)

In [ ]:
#Methods, decoder effect sizes of ED Fig 8b (the N and sigma^2 sweeps): both decoders in the full space
#vs the estimated subspace, paired per held-out condition, mean(delta)/std(delta) and the ROC AUC, at
#the C with the lowest held-out error per space (as the ED Fig 8 panels) and at the data's fixed C;
#10 seeds per cell, cached in cache/simulation_effect -> simulation_effect_sizes.csv
for cell in sim.EFFECT_CELLS:
    for seed in range(10):
        sim.effect_sizes_cached(project_root, cell, seed)
effect_sizes = sim.effect_size_table(project_root)
effect_sizes[['cell', 'n_seeds', 'N', 'inv_snr', 'svm_d_paired_mean', 'svm_auc_mean', 'ori_d_paired_mean', 'ori_auc_mean']].round(2)

### measured parameters of the recorded populations (ED Fig 10a)
the generator parameters measurable from the data, per population and neuron subsample: the radius spread
delta/r of the tuned planes, the per-neuron noise-to-signal ratio and its spread; needs the single-trial
caches of every row of `model_params.csv` (`trial_caches_all`, also used by `plot_extended_figures.ipynb`)

In [ ]:
trial_caches_all(model_df, params_df, project_root)

In [ ]:
datafit_params = sim.datafit_params(project_root)      # results_extended/data_fit/data_fit_params.csv
pd.DataFrame(sim.area_params(project_root)).T.round(3)   # the means per population that the simulations use

### calibration simulations based on pixels (ED Fig 9c-h)
the pooled centered CC_norm of the equivariance model over an (N, d) grid. First the recorded populations:
the pixels with the injected noise and noiseless, the pooled visual areas of the three mice up to their
reliable-neuron counts (10,698 / 3,100 / 1,696 / 855), and AlexNet conv1-5 up to their reliable unit counts
(noisy and the same units noiseless). Each (population, N) is a long computation (the N = 8000 points take
hours); every (N, d) point is cached and skipped afterwards

In [ ]:
#single-trial caches: the pooled mouse areas (from the per-mouse caches written above), the noiseless
#pixels and AlexNet layers (params/fixed_params_eqv_snr0.csv) and the noiseless responses of exactly
#the units in the noisy AlexNet caches
sim.build_pooled_caches(project_root)
sim.build_noiseless_caches(project_root)
sim.build_noiseless_rel_caches(project_root)

In [ ]:
#the (N, d) surfaces of the recorded populations: N as in the figure, d = 6 ... 800 (d < N), 2 disjoint
#neuron subsamples per (N, d) (4 for AL, whose 855 neurons give 3 subsamples at N = 250)
sim.ccnorm_surface(project_root, 'pixels', 'pixels', plot_sim.N_RAMP)
sim.ccnorm_surface(project_root, 'pixels', 'pixels', plot_sim.N_RAMP, noise_model='noiseless')
for area, n_list in plot_sim.AREA_N_LEVELS.items():
    d_list = sim.D_LIST if area == 'V1' else sim.D_LIST[:-1]
    sim.ccnorm_surface(project_root, 'mouse_pooled', area, n_list, d_list, n_max=4 if area == 'AL' else 2)
for layer, n_list in plot_sim.CONV_N_LEVELS.items():
    sim.ccnorm_surface(project_root, 'alexnet', layer, n_list)
    sim.ccnorm_surface(project_root, 'alexnet', layer, n_list, noise_model='noiseless_rel')

the simulated populations built from the equivariant structure of the pixels (ED Fig 9d right): the
structure is measured on the noise-free pixel responses at full rank (`sim.band_structure`; Methods), then
populations of 1e5 neurons are generated with that structure at an equivariant signal fraction f = 1 - iid
and equivariant dimension R (all 1362 dimensions, or the top 500 / 20 by variance), with the measured radius
spread and the pixels' noise, and run through the same (N, d) grid (`sim.run_matched_populations`)

In [ ]:
#the equivariant structure of the noise-free pixels: dimensions per rotation frequency (Methods: 90 invariant,
#84 tuned planes, 93 / 91 / 92 / 92 / 92 / 92 higher-order planes) -> results_extended/calibration/band_structure_pixels.json
bands, meta = sim.band_structure(project_root, 'pixels')
pd.DataFrame({f: dict(dims=bands[f]['count'], share=round(bands[f]['share'], 3)) for f in bands}).T

In [ ]:
sim.run_matched_populations(project_root, 'pixels', iid_list=(0.0,))                     # f = 1, all dimensions
sim.run_matched_populations(project_root, 'pixels', iid_list=(0.0, 0.5), rank=500)       # f = 1 / 0.5, R = 500
sim.run_matched_populations(project_root, 'pixels', iid_list=(0.0, 0.5, 0.9), rank=20)   # f = 1 / 0.5 / 0.1, R = 20

In [ ]:
#ED Fig 9c-h -> results_extended/calibration/ccnorm_vs_d_{pixels,conv1,conv_layers,V1,areas}.pdf,
#ccnorm_peak_vs_N_{pixels,pixel_simulations,conv_layers,areas}.pdf
plot_sim.plot_ed9_panels(project_root)

### fit of simulations to data (ED Fig 10)
for every recorded population a simulated population is built from its measured equivariant structure (the
noisy responses with the split-trial estimator; Methods) and its measured noise and radius spread; the
equivariant signal fraction f = 1 - iid is the one whose (N, d) surface deviates least from the recorded one
(d = 20 ... 200), read off a sweep over iid (`sim.best_iid`, `sim.fill_swept` -> `params/datafit_swept_iid.csv`).
The sweeps build one 1e5-neuron population per iid value for the mouse areas and are the heaviest step of
this notebook (cluster)

In [ ]:
#the equivariant structure of every population and the sweeps over the non-equivariant fraction
sweep_iids = {name: ((0.0, 0.1, 0.2, 0.3, 0.4) if name == 'pixels' else tuple(np.round(np.arange(0, 1, 0.1), 1)))
              for name in sim.TARGET_ORDER}
for name in sim.TARGET_ORDER:
    sim.band_structure(project_root, name)

In [ ]:
for name in sim.TARGET_ORDER:
    sim.run_matched_populations(project_root, name, iid_list=sweep_iids[name])

In [ ]:
swept = sim.fill_swept(project_root)                            # -> params/datafit_swept_iid.csv

In [ ]:
#ED Fig 10b: the structure-matched populations at their swept f through the standard pipeline (N = 250,
#d = 20, 2 trials, with the decoders), 10 seeds per population; cached in cache/simulation_datafit_sized
for name in sim.TARGET_ORDER:
    for seed in range(10):
        sim.run_target_cached(project_root, name, seed)
datafit_sim = sim.datafit_table(project_root)                   # results_extended/data_fit/data_fit_sim.csv
datafit_sim.groupby('area')[['d_inv', 'd_tun', 'cc_norm_equivariance_c', 'svm_err_eqv_C1', 'ori_err_eqv']].mean().loc[sim.TARGET_ORDER].round(3)

In [ ]:
#ED Fig 10a-b -> results_extended/data_fit/datafit_{delta_over_r,snr_hetero,sigma_calib,f_calib}.pdf and
#datafit_sim_{dim,cc,svm1C,ori}.pdf; the SI Fig 7 calibration panels gain their data-matched points once
#data_fit_sim.csv exists (re-run plot_calibration above)
plot_sim.plot_datafit_params(project_root)
plot_sim.plot_datafit_simulations(project_root)
plot_sim.plot_calibration(summary, results_simulation)